# Lab 04 — Schema Enforcement

This notebook proves that a Delta table rejects incompatible writes when schema evolution is not enabled. It also demonstrates a controlled rescue pattern that preserves rejected payloads without silently changing the analytics table.

## Learning objectives

- create a Delta table with an explicit, stable schema;
- append a compatible batch successfully;
- observe Delta rejecting an incompatible data type;
- observe Delta rejecting an unexpected source column;
- prove rejected writes do not partially modify the target;
- compare storage enforcement with an explicit data contract;
- preserve diagnostics in a quarantine table.

> The negative tests intentionally cause Delta write errors, but the exceptions are caught and validated. Therefore, a successful notebook run includes two expected rejections.

## 1. Load shared configuration

The configuration notebook supplies the catalog, schema, volume paths, table names, active contract version, and validation settings. Notebook 04 must have completed successfully because this notebook uses the curated Silver transaction table as its trusted input.

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DecimalType, LongType, StringType, StructField, StructType, TimestampType,
)

schema_demo_table = table_names["schema_demo"]
schema_quarantine_table = f"{catalog}.{schema}.lab04_schema_enforcement_quarantine"

# Notebook 08 intentionally demonstrates strict enforcement of contract v1.
# The runtime widget may be v1 or v2, but that must not change the baseline
# contract used by this specific enforcement experiment.
if "contract_v1" not in globals() or "contract_v2" not in globals():
    raise NameError(
        "contract_v1/contract_v2 are not loaded. Update lab04_00_config so it "
        "loads contracts/online_retail_v1.yml and online_retail_v2.yml."
    )

enforcement_contract = contract_v1
enforcement_contract_version = f"v{enforcement_contract['contract']['version']}"

v1_contract_columns = {
    item["name"]: item
    for item in contract_v1["schema"]["columns"]
}
v2_contract_columns = {
    item["name"]: item
    for item in contract_v2["schema"]["columns"]
}
v2_added_columns = sorted(set(v2_contract_columns) - set(v1_contract_columns))

if enforcement_contract_version != "v1":
    raise AssertionError(
        f"Notebook 08 expects contract v1 as the enforcement baseline; "
        f"loaded {enforcement_contract_version}."
    )
if contract_v1["schema"].get("allow_additional_columns", False):
    raise AssertionError("Contract v1 must forbid unapproved additional columns.")
if "loyalty_tier" not in v2_added_columns:
    raise AssertionError(
        "Contract v2 must introduce loyalty_tier for the unexpected-column test."
    )

contract_inventory_df = spark.createDataFrame(
    [
        (
            "v1",
            str(contract_v1["contract"]["version"]),
            contract_v1["contract"]["status"],
            len(contract_v1["schema"]["columns"]),
        ),
        (
            "v2",
            str(contract_v2["contract"]["version"]),
            contract_v2["contract"]["status"],
            len(contract_v2["schema"]["columns"]),
        ),
    ],
    ["contract", "version", "status", "column_count"],
)

print(f"Silver source: {silver_table}")
print(f"Enforced target: {schema_demo_table}")
print(f"Quarantine target: {schema_quarantine_table}")
print(f"Configured schema policy: {schema_policy}")
print(f"Runtime widget contract: {contract_version}")
print(f"Notebook 08 enforcement baseline: {enforcement_contract_version}")
print(f"Columns introduced by v2: {v2_added_columns}")
display(contract_inventory_df)


Silver source: dbr_dev.parvinbadalov.lab04_silver_transactions
Enforced target: dbr_dev.parvinbadalov.lab04_schema_demo
Quarantine target: dbr_dev.parvinbadalov.lab04_schema_enforcement_quarantine
Configured schema policy: fail
Runtime widget contract: v1
Notebook 08 enforcement baseline: v1
Columns introduced by v2: ['loyalty_tier', 'sales_channel']


contract,version,status,column_count
v1,1,active,8
v2,2,proposed,10


## 2. Verify the Silver prerequisite

Schema tests are meaningful only when the source is already clean and typed. The check below fails early if the Silver MERGE notebook has not created a populated table.

In [0]:
if not spark.catalog.tableExists(silver_table):
    raise FileNotFoundError(
        f"Silver table {silver_table} does not exist. "
        "Run lab04_04_silver_merge.ipynb first."
    )

required_silver_columns = {
    "transaction_line_id", "invoice_no", "stock_code", "quantity",
    "unit_price", "invoice_timestamp", "country",
    "source_batch_id", "quality_contract_version",
}
silver_source_df = spark.table(silver_table)
missing_silver_columns = sorted(required_silver_columns - set(silver_source_df.columns))
if missing_silver_columns:
    raise AssertionError(f"Silver is missing required columns: {missing_silver_columns}")

silver_source_count = silver_source_df.count()
if silver_source_count == 0:
    raise ValueError(f"Silver table {silver_table} is empty.")

print(f"✅ Silver prerequisite passed: {silver_source_count:,} rows available.")

✅ Silver prerequisite passed: 315,101 rows available.


## 3. Define the enforced schema

The schema is explicit rather than inferred. This is the storage boundary for the demonstration: names and data types must match exactly unless a later notebook deliberately enables evolution.

The target is recreated on each run so the demonstration is deterministic and does not accumulate duplicate sample rows. This reset affects only `lab04_schema_demo`, never the production Silver table.

In [0]:
enforced_schema = StructType([
    StructField("transaction_line_id", StringType(), False),
    StructField("invoice_no", StringType(), False),
    StructField("stock_code", StringType(), False),
    StructField("quantity", LongType(), False),
    StructField("unit_price", DecimalType(18, 4), False),
    StructField("invoice_timestamp", TimestampType(), False),
    StructField("country", StringType(), False),
    StructField("source_batch_id", StringType(), False),
    StructField("contract_version", StringType(), False),
    StructField("accepted_at", TimestampType(), False),
])

spark.sql(f"DROP TABLE IF EXISTS {schema_demo_table}")
spark.sql(
    f"""
    CREATE TABLE {schema_demo_table} (
        transaction_line_id STRING NOT NULL,
        invoice_no STRING NOT NULL,
        stock_code STRING NOT NULL,
        quantity BIGINT NOT NULL,
        unit_price DECIMAL(18,4) NOT NULL,
        invoice_timestamp TIMESTAMP NOT NULL,
        country STRING NOT NULL,
        source_batch_id STRING NOT NULL,
        contract_version STRING NOT NULL,
        accepted_at TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'quality.contract' = 'online_retail_schema_enforcement',
        'quality.contract.version' = '{enforcement_contract_version}',
        'quality.contract.source' = 'contracts/online_retail_v1.yml',
        'delta.enableChangeDataFeed' = 'true'
    )
    """
)

spark.table(schema_demo_table).printSchema()
print(
    f"✅ Explicit Delta target created from the {enforcement_contract_version} "
    f"enforcement baseline: {schema_demo_table}"
)


root
 |-- transaction_line_id: string (nullable = false)
 |-- invoice_no: string (nullable = false)
 |-- stock_code: string (nullable = false)
 |-- quantity: long (nullable = false)
 |-- unit_price: decimal(18,4) (nullable = false)
 |-- invoice_timestamp: timestamp (nullable = false)
 |-- country: string (nullable = false)
 |-- source_batch_id: string (nullable = false)
 |-- contract_version: string (nullable = false)
 |-- accepted_at: timestamp (nullable = false)

✅ Explicit Delta target created from the v1 enforcement baseline: dbr_dev.parvinbadalov.lab04_schema_demo


## 4. Build a compatible batch

Five deterministic Silver rows are projected into the exact target column order and data types. This batch represents a producer that follows contract v1.

In [0]:
compatible_df = (
    silver_source_df
    .orderBy("transaction_line_id")
    .limit(5)
    .select(
        F.col("transaction_line_id").cast("string"),
        F.col("invoice_no").cast("string"),
        F.col("stock_code").cast("string"),
        F.col("quantity").cast("long"),
        F.col("unit_price").cast("decimal(18,4)"),
        F.col("invoice_timestamp").cast("timestamp"),
        F.col("country").cast("string"),
        F.col("source_batch_id").cast("string"),
        F.col("quality_contract_version").cast("string").alias("contract_version"),
        F.current_timestamp().alias("accepted_at"),
    )
)

compatible_count = compatible_df.count()
if compatible_count != 5:
    raise AssertionError(f"Expected 5 compatible sample rows, found {compatible_count}.")

compatible_df.printSchema()
display(compatible_df)

root
 |-- transaction_line_id: string (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: decimal(18,4) (nullable = true)
 |-- invoice_timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- source_batch_id: string (nullable = true)
 |-- contract_version: string (nullable = true)
 |-- accepted_at: timestamp (nullable = false)



transaction_line_id,invoice_no,stock_code,quantity,unit_price,invoice_timestamp,country,source_batch_id,contract_version,accepted_at
00001c8394e60696d3ae8dd2517c6755ff3fadfb46b12276ef4574ae66efafa2,580162,84978,2,1.2500,2011-12-02T10:52:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:10.552Z
000040ca23d38cc54100c36a99f626d451a1740a6af4c0b0b0dca5a1399b45d2,540647,21190,2,1.6500,2011-01-10T14:57:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:10.552Z
00005b3c325c78c5d1d95b540a57613a6d4c1d61d67813f19059db47013880dc,557016,23268,1,1.4500,2011-06-16T12:29:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:10.552Z
00008cceb3ba0f98407fdaf32e2714a9dea044bdf03489e2285e428854c310c6,567074,20726,5,1.6500,2011-09-16T12:22:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:10.552Z
0000d3d554c456640a1a557899acb1e48d98528324ae82d2c069e4c7464c70c0,561534,22434,8,1.9500,2011-07-28T09:45:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:10.552Z


## 5. Accept the matching write

No evolution option is supplied. Delta accepts the append because the incoming names and types conform to the target schema.

In [0]:
(
    compatible_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(schema_demo_table)
)

baseline_count = spark.table(schema_demo_table).count()
if baseline_count != compatible_count:
    raise AssertionError(
        f"Compatible append mismatch: expected {compatible_count}, found {baseline_count}."
    )

print(f"✅ Compatible append accepted: {baseline_count} rows in target.")
display(spark.table(schema_demo_table).orderBy("transaction_line_id"))

✅ Compatible append accepted: 5 rows in target.


transaction_line_id,invoice_no,stock_code,quantity,unit_price,invoice_timestamp,country,source_batch_id,contract_version,accepted_at
00001c8394e60696d3ae8dd2517c6755ff3fadfb46b12276ef4574ae66efafa2,580162,84978,2,1.2500,2011-12-02T10:52:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:11.718Z
000040ca23d38cc54100c36a99f626d451a1740a6af4c0b0b0dca5a1399b45d2,540647,21190,2,1.6500,2011-01-10T14:57:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:11.718Z
00005b3c325c78c5d1d95b540a57613a6d4c1d61d67813f19059db47013880dc,557016,23268,1,1.4500,2011-06-16T12:29:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:11.718Z
00008cceb3ba0f98407fdaf32e2714a9dea044bdf03489e2285e428854c310c6,567074,20726,5,1.6500,2011-09-16T12:22:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:11.718Z
0000d3d554c456640a1a557899acb1e48d98528324ae82d2c069e4c7464c70c0,561534,22434,8,1.9500,2011-07-28T09:45:00.000Z,United Kingdom,initial,v1,2026-08-10T22:21:11.718Z


## 6. Reject an incompatible data type

The `quantity` column is deliberately changed from `BIGINT` to a nested `STRUCT`. This is not a safe cast or a widening conversion, so Delta must reject the append. The exception is evidence, not an unexpected notebook failure.

In [0]:
type_mismatch_df = compatible_df.withColumn(
    "quantity",
    F.struct(F.col("quantity").alias("value")),
)

type_mismatch_rejected = False
type_mismatch_error = None

try:
    (
        type_mismatch_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(schema_demo_table)
    )
except Exception as exc:
    type_mismatch_rejected = True
    type_mismatch_error = str(exc).splitlines()[0][:500]

if not type_mismatch_rejected:
    raise AssertionError("Delta unexpectedly accepted STRUCT quantity into BIGINT quantity.")

count_after_type_rejection = spark.table(schema_demo_table).count()
if count_after_type_rejection != baseline_count:
    raise AssertionError("Rejected type-mismatch write partially modified the target.")

print("✅ Expected type-mismatch rejection observed.")
print(type_mismatch_error)
print(f"Target remains unchanged at {count_after_type_rejection} rows.")

✅ Expected type-mismatch rejection observed.
[DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "quantity" due to data type mismatch: cannot cast "STRUCT<value: BIGINT>" to "BIGINT". SQLSTATE: 42K09;
Target remains unchanged at 5 rows.


## 7. Reject an unexpected column

The repository **contract v1** is the fixed baseline for this enforcement demonstration. It forbids arbitrary additional columns. `loyalty_tier` is deliberately taken from the set of fields introduced by **contract v2**, then written without `mergeSchema`. Delta rejects it rather than changing the table silently.

> The `contract_version` widget controls the runtime pipeline selection. It does **not** rewrite this historical v1 → v2 enforcement demonstration.


In [0]:
unexpected_column_name = "loyalty_tier"
unexpected_column_value = (
    v2_contract_columns[unexpected_column_name]
    .get("allowed_values", ["STANDARD"])[0]
)

unexpected_column_df = compatible_df.withColumn(
    unexpected_column_name,
    F.lit(unexpected_column_value),
)
unexpected_column_rejected = False
unexpected_column_error = None

try:
    (
        unexpected_column_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(schema_demo_table)
    )
except Exception as exc:
    unexpected_column_rejected = True
    unexpected_column_error = str(exc).splitlines()[0][:500]

if not unexpected_column_rejected:
    raise AssertionError(
        f"Delta unexpectedly accepted the unapproved {unexpected_column_name} column."
    )

count_after_column_rejection = spark.table(schema_demo_table).count()
if count_after_column_rejection != baseline_count:
    raise AssertionError("Rejected unexpected-column write partially modified the target.")

print(
    f"✅ Expected unexpected-column rejection observed: "
    f"{unexpected_column_name} is not part of {enforcement_contract_version}."
)
print(unexpected_column_error)
print(f"Target remains unchanged at {count_after_column_rejection} rows.")


✅ Expected unexpected-column rejection observed: loyalty_tier is not part of v1.
[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
Target remains unchanged at 5 rows.


## 8. Validate the contract before writing

Delta enforcement protects the table at write time. A data contract gives an earlier and clearer failure by comparing expected and incoming column names and types before storage is attempted. This is preferable to relying only on engine error messages.

In [0]:
# Technical Delta contract used by this isolated enforcement table.
# The repository YAML is the governance source of truth for which evolution is approved.
expected_contract = {
    field.name: field.dataType.simpleString()
    for field in enforced_schema.fields
}

def compare_to_contract(dataframe, expected):
    actual = {
        field.name: field.dataType.simpleString()
        for field in dataframe.schema.fields
    }
    missing = sorted(set(expected) - set(actual))
    unexpected = sorted(set(actual) - set(expected))
    type_mismatches = {
        name: {"expected": expected[name], "actual": actual[name]}
        for name in sorted(set(expected) & set(actual))
        if expected[name] != actual[name]
    }
    return {
        "valid": not (missing or unexpected or type_mismatches),
        "missing_columns": missing,
        "unexpected_columns": unexpected,
        "type_mismatches": type_mismatches,
    }

compatible_contract_result = compare_to_contract(
    compatible_df,
    expected_contract,
)
type_contract_result = compare_to_contract(
    type_mismatch_df,
    expected_contract,
)
column_contract_result = compare_to_contract(
    unexpected_column_df,
    expected_contract,
)

if not compatible_contract_result["valid"]:
    raise AssertionError(
        f"Compatible batch failed contract validation: "
        f"{compatible_contract_result}"
    )
if type_contract_result["valid"] or column_contract_result["valid"]:
    raise AssertionError(
        "A deliberately invalid batch unexpectedly passed the contract."
    )

yaml_governance_checks = [
    (
        "v1_forbids_additional_columns",
        not contract_v1["schema"].get("allow_additional_columns", False),
        str(contract_v1["schema"].get("allow_additional_columns")),
    ),
    (
        "loyalty_tier_absent_from_v1",
        "loyalty_tier" not in v1_contract_columns,
        "v1 columns loaded from YAML",
    ),
    (
        "loyalty_tier_approved_in_v2",
        "loyalty_tier" in v2_contract_columns,
        "v2 columns loaded from YAML",
    ),
]

contract_results_df = spark.createDataFrame(
    [
        ("compatible_batch", str(compatible_contract_result)),
        ("type_mismatch_batch", str(type_contract_result)),
        ("unexpected_column_batch", str(column_contract_result)),
    ],
    ["test_case", "contract_result"],
)
display(contract_results_df)

yaml_governance_df = spark.createDataFrame(
    yaml_governance_checks,
    ["governance_check", "passed", "evidence"],
)
display(yaml_governance_df)

if yaml_governance_df.filter(~F.col("passed")).count():
    raise AssertionError("Repository YAML governance checks failed.")

print("✅ Delta and YAML contract validation classified all test cases correctly.")


test_case,contract_result
compatible_batch,"{'valid': True, 'missing_columns': [], 'unexpected_columns': [], 'type_mismatches': {}}"
type_mismatch_batch,"{'valid': False, 'missing_columns': [], 'unexpected_columns': [], 'type_mismatches': {'quantity': {'expected': 'bigint', 'actual': 'struct'}}}"
unexpected_column_batch,"{'valid': False, 'missing_columns': [], 'unexpected_columns': ['loyalty_tier'], 'type_mismatches': {}}"


governance_check,passed,evidence
v1_forbids_additional_columns,true,False
loyalty_tier_absent_from_v1,true,v1 columns loaded from YAML
loyalty_tier_approved_in_v2,true,v2 columns loaded from YAML


✅ Delta and YAML contract validation classified all test cases correctly.


## 9. Preserve rejected payloads in quarantine

`fail` protects the target, but it does not retain the rejected records. A controlled rescue policy writes the original record and diagnostics to a separate quarantine table. The Silver schema remains unchanged, and operators can inspect or repair the rejected payload later.

This is an explicit application-level rescue pattern. It is conceptually similar to Auto Loader's `_rescued_data`, but it is implemented here for controlled Delta writes.

In [0]:
type_rescue_df = (
    type_mismatch_df
    .select(
        F.sha2(
            F.concat_ws("||", "transaction_line_id", F.lit("TYPE_MISMATCH")),
            256,
        ).alias("quarantine_id"),
        F.col("transaction_line_id").alias("source_record_id"),
        F.lit("TYPE_MISMATCH").alias("violation_code"),
        F.lit("quantity expected BIGINT but received STRUCT").alias("violation_detail"),
        F.to_json(
            F.struct(*[F.col(name) for name in type_mismatch_df.columns])
        ).alias("source_payload"),
        F.to_json(F.struct(F.col("quantity"))).alias("_rescued_data"),
        F.lit(enforcement_contract_version).alias("expected_contract_version"),
        F.current_timestamp().alias("quarantined_at"),
    )
)

column_rescue_df = (
    unexpected_column_df
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                "transaction_line_id",
                F.lit("UNEXPECTED_COLUMN"),
            ),
            256,
        ).alias("quarantine_id"),
        F.col("transaction_line_id").alias("source_record_id"),
        F.lit("UNEXPECTED_COLUMN").alias("violation_code"),
        F.lit(
            f"{unexpected_column_name} is not approved by "
            f"contract {enforcement_contract_version}"
        ).alias("violation_detail"),
        F.to_json(
            F.struct(*[F.col(name) for name in unexpected_column_df.columns])
        ).alias("source_payload"),
        F.to_json(
            F.struct(F.col(unexpected_column_name))
        ).alias("_rescued_data"),
        F.lit(enforcement_contract_version).alias("expected_contract_version"),
        F.current_timestamp().alias("quarantined_at"),
    )
)

quarantine_demo_df = type_rescue_df.unionByName(column_rescue_df)

(
    quarantine_demo_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(schema_quarantine_table)
)

quarantine_count = spark.table(schema_quarantine_table).count()
expected_quarantine_count = compatible_count * 2
if quarantine_count != expected_quarantine_count:
    raise AssertionError(
        f"Expected {expected_quarantine_count} quarantine rows, "
        f"found {quarantine_count}."
    )

display(
    spark.table(schema_quarantine_table)
    .select(
        "source_record_id",
        "violation_code",
        "violation_detail",
        "_rescued_data",
        "expected_contract_version",
        "quarantined_at",
    )
    .orderBy("source_record_id", "violation_code")
)
print(f"✅ Controlled rescue preserved {quarantine_count} rejected payloads.")


source_record_id,violation_code,violation_detail,_rescued_data,expected_contract_version,quarantined_at
00001c8394e60696d3ae8dd2517c6755ff3fadfb46b12276ef4574ae66efafa2,TYPE_MISMATCH,quantity expected BIGINT but received STRUCT,"{""quantity"":{""value"":2}}",v1,2026-08-10T22:21:18.077Z
00001c8394e60696d3ae8dd2517c6755ff3fadfb46b12276ef4574ae66efafa2,UNEXPECTED_COLUMN,loyalty_tier is not approved by contract v1,"{""loyalty_tier"":""STANDARD""}",v1,2026-08-10T22:21:18.077Z
000040ca23d38cc54100c36a99f626d451a1740a6af4c0b0b0dca5a1399b45d2,TYPE_MISMATCH,quantity expected BIGINT but received STRUCT,"{""quantity"":{""value"":2}}",v1,2026-08-10T22:21:18.077Z
000040ca23d38cc54100c36a99f626d451a1740a6af4c0b0b0dca5a1399b45d2,UNEXPECTED_COLUMN,loyalty_tier is not approved by contract v1,"{""loyalty_tier"":""STANDARD""}",v1,2026-08-10T22:21:18.077Z
00005b3c325c78c5d1d95b540a57613a6d4c1d61d67813f19059db47013880dc,TYPE_MISMATCH,quantity expected BIGINT but received STRUCT,"{""quantity"":{""value"":1}}",v1,2026-08-10T22:21:18.077Z
00005b3c325c78c5d1d95b540a57613a6d4c1d61d67813f19059db47013880dc,UNEXPECTED_COLUMN,loyalty_tier is not approved by contract v1,"{""loyalty_tier"":""STANDARD""}",v1,2026-08-10T22:21:18.077Z
00008cceb3ba0f98407fdaf32e2714a9dea044bdf03489e2285e428854c310c6,TYPE_MISMATCH,quantity expected BIGINT but received STRUCT,"{""quantity"":{""value"":5}}",v1,2026-08-10T22:21:18.077Z
00008cceb3ba0f98407fdaf32e2714a9dea044bdf03489e2285e428854c310c6,UNEXPECTED_COLUMN,loyalty_tier is not approved by contract v1,"{""loyalty_tier"":""STANDARD""}",v1,2026-08-10T22:21:18.077Z
0000d3d554c456640a1a557899acb1e48d98528324ae82d2c069e4c7464c70c0,TYPE_MISMATCH,quantity expected BIGINT but received STRUCT,"{""quantity"":{""value"":8}}",v1,2026-08-10T22:21:18.077Z
0000d3d554c456640a1a557899acb1e48d98528324ae82d2c069e4c7464c70c0,UNEXPECTED_COLUMN,loyalty_tier is not approved by contract v1,"{""loyalty_tier"":""STANDARD""}",v1,2026-08-10T22:21:18.077Z


✅ Controlled rescue preserved 10 rejected payloads.


## 10. Final validation and evidence

The summary proves that enforcement was atomic: only the compatible batch reached the target, both incompatible batches were rejected, and rejected content was retained separately. Save this result and one expected error message as README evidence.

In [0]:
final_target_count = spark.table(schema_demo_table).count()
final_target_columns = spark.table(schema_demo_table).columns
target_schema_unchanged = final_target_columns == [
    field.name for field in enforced_schema.fields
]

validation_rows = [
    (
        "compatible_write_accepted",
        compatible_count == baseline_count,
        str(baseline_count),
    ),
    (
        "type_mismatch_rejected",
        type_mismatch_rejected,
        type_mismatch_error or "",
    ),
    (
        "unexpected_column_rejected",
        unexpected_column_rejected,
        unexpected_column_error or "",
    ),
    (
        "target_count_unchanged_after_rejections",
        final_target_count == baseline_count,
        str(final_target_count),
    ),
    (
        "target_schema_unchanged",
        target_schema_unchanged,
        ", ".join(final_target_columns),
    ),
    (
        "rejected_payloads_quarantined",
        quarantine_count == expected_quarantine_count,
        str(quarantine_count),
    ),
    (
        "yaml_v1_loaded",
        enforcement_contract_version == "v1",
        enforcement_contract_version,
    ),
    (
        "yaml_v1_forbids_additional_columns",
        not contract_v1["schema"].get("allow_additional_columns", False),
        str(contract_v1["schema"].get("allow_additional_columns")),
    ),
    (
        "yaml_v2_declares_loyalty_tier",
        "loyalty_tier" in v2_contract_columns,
        ", ".join(v2_added_columns),
    ),
]
validation_df = spark.createDataFrame(
    validation_rows,
    ["validation", "passed", "evidence"],
)
display(validation_df)

failed_validations = validation_df.filter(~F.col("passed")).count()
if run_validation and failed_validations != 0:
    raise AssertionError(
        f"Schema enforcement validation failed: "
        f"{failed_validations} checks did not pass."
    )

display(
    spark.sql(f"DESCRIBE HISTORY {schema_demo_table}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
)
print("✅ Schema enforcement demonstration completed successfully.")


validation,passed,evidence
compatible_write_accepted,true,5
type_mismatch_rejected,true,"[DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve ""quantity"" due to data type mismatch: cannot cast ""STRUCT"" to ""BIGINT"". SQLSTATE: 42K09;"
unexpected_column_rejected,true,[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
target_count_unchanged_after_rejections,true,5
target_schema_unchanged,true,"transaction_line_id, invoice_no, stock_code, quantity, unit_price, invoice_timestamp, country, source_batch_id, contract_version, accepted_at"
rejected_payloads_quarantined,true,10
yaml_v1_loaded,true,v1
yaml_v1_forbids_additional_columns,true,False
yaml_v2_declares_loyalty_tier,true,"loyalty_tier, sales_channel"


version,timestamp,operation,operationMetrics
1,2026-08-10T22:21:13.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 3708)"
0,2026-08-10T22:21:10.000Z,CREATE TABLE,Map()


✅ Schema enforcement demonstration completed successfully.


## What this notebook proved

| Scenario | Result | Correct operational response |
|---|---|---|
| Matching names and types | Accepted | Write to the governed Delta target |
| Incompatible type | Rejected | Correct the producer or quarantine the payload |
| Unexpected column | Rejected | Review and approve a new contract version |
| Rescue policy | Target unchanged | Preserve raw payload and diagnostics separately |

Schema enforcement is the safe default. It prevents accidental drift but does not decide whether a new field is useful. That decision belongs to a data contract and an intentional evolution process.

## Next notebook

Continue with **`lab04_09_schema_evolution.ipynb`**. It will introduce contract v2 and demonstrate controlled Delta evolution with `mergeSchema` and `autoMerge`, while distinguishing approved additive changes from unsafe type changes.